
# Transform Drivers Data

1. Read bronze drivers table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (driverId -> driver_id, dateOfbirth -> date_of_birth)
4. Concatenate name.givenName and name.familyName to create a new column called driver_name and transform the value to Title Case
5. Remove duplicate records
6. Transform values of columns nationality to Title Case
7. Write the transformed data to silver drivers table

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%run ../00-Common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"


### Step 1 - Read bronze constructors table

In [0]:
# circuits_df = spark.table(bronze_table)
drivers_df = spark.read.table(bronze_table)

### Step 2 - Keep only the columns required for analytics (Drop url column)



In [0]:
drivers_selected_df = drivers_df.drop(
    col("url")   
)


### Step 3 - Standardise column names using snake_case (driverId -> driver_id, dateOfbirth -> date_of_birth)

In [0]:
drivers_renamed_df = drivers_selected_df\
    .withColumnsRenamed(
        {
            "driverId":"driver_id",
            "dateOfBirth":"date_of_birth"
        }
    )

### Step 4 - Concatenate name.givenName and name.familyName to create a new column called driver_name and transform the value to Title Case

In [0]:
drivers_concatenated_df = drivers_renamed_df.withColumn(
        "driver_name", 
        initcap(concat_ws(" ",drivers_renamed_df.name.givenName, drivers_renamed_df.name.familyName))
    ).drop("name")


### Step 5 - Remove Duplicate Records


In [0]:
drivers_distinct_df = drivers_concatenated_df.dropDuplicates(["driver_id"])


### Step  6 - Transform Value of column nationality to Title Case


In [0]:
drivers_final_df = (
    drivers_distinct_df
    .withColumn('nationality', initcap('nationality'))
)



### Step  7 - Write the transformed data to silver drivers table

In [0]:
(
    drivers_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))